In [1]:
## Author: Fabio Pintore
## Institute: INAF/IASF Palermo
#
# The code is based on Gammapy v1.3 and allows for an end-to-end simulation and
# data analysis. It permits to fit models and obtain the spectra, lightcurves, counts
# maps and TS maps.
#
#
# For any request, please write to fabio.pintore@inaf.it
#

In [2]:
import numpy as np
from pathlib import Path

from astropy.table import Table
import astropy.units as u
from astropy.coordinates import Angle, SkyCoord
from astropy.time import Time
from regions import CircleSkyRegion, PointSkyRegion
import matplotlib.pyplot as plt
from gammapy.data import FixedPointingInfo, Observation, observatory_locations
from gammapy.datasets import MapDataset, MapDatasetEventSampler
from gammapy.irf import load_irf_dict_from_file
from gammapy.makers import MapDatasetMaker
from gammapy.maps import MapAxis, RegionNDMap, WcsGeom, TimeMapAxis
from gammapy.modeling.models import (
    ConstantSpectralModel,
    FoVBackgroundModel,
    LightCurveTemplateTemporalModel,
    PointSpatialModel,
    PowerLawSpectralModel,
    SkyModel,
    Models
)

/Users/jarred/Work/sensipy/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import gammapy
gammapy.__version__

'2.0'

# Make the energy-dependent model in a Gammapy compliant format

Open the model, if you have the fits file:

In [ ]:
energies = Table.read("catO5_301.fits", format="fits", hdu=1)
times = Table.read("catO5_301.fits", format="fits", hdu=2)
spectra = Table.read("catO5_301.fits", format="fits", hdu=3)

print(f"Number of energies: {len(energies)}. \nNumber of times: {len(times)}")

Define three time bins before the T0 of the events. These time bins will be set at zero flux:

In [ ]:
delta = (np.log10(times[-1][1]) - np.log10(times[0][0])) / len(times)
n_time_extra = 3
new_times = np.logspace(np.log10(times[0][0]) - delta * n_time_extra, np.log10(times[-1][1]), 71 + n_time_extra)

Let's create the gammapy format of the model. Define a soure position and the gammapy axes for times and energies, and then fill only the epochs with flux different from zero:

In [ ]:
# source position
pointing_position = SkyCoord("274.219 deg", "1.492 deg", frame="icrs")
position = FixedPointingInfo(fixed_icrs=pointing_position.icrs)

# time axis
#time_axis = MapAxis.from_bounds(new_times[0], new_times[-1], nbin=73, name="time", interp="log")
time_axis = MapAxis.from_edges(new_times, unit="s", name="time", interp="log")

# energy axis
energy_axis = MapAxis.from_nodes(
    energies['Energies'], unit=energies['Energies'].unit, name="energy"
)

# create the spectrum in the new time range
spec_mod = np.lib.recfunctions.structured_to_unstructured(spectra.as_array())
newspec = np.zeros( (len(time_axis.center) , len(energy_axis.center)))

# fill the spectrum
newspec[n_time_extra:,:] = spec_mod

Let's plot a given time interval:

In [ ]:
plt.plot(energy_axis.center, newspec[42])
plt.loglog()

Make the Gammapy maps of the model. Please, be careful with the spectral units:

In [ ]:
# create the RegionNDMap containing fluxes
m = RegionNDMap.create(
    region=PointSkyRegion(center=pointing_position),
    axes=[energy_axis, time_axis],
    unit="cm-2 s-1 GeV-1",
)

# to compute the spectra as a function of time we extract the coordinates of the geometry
coords = m.geom.get_coord(sparse=True)

# We reshape the spectrum array to perform broadcasting
newspec = newspec.reshape(m.data.shape)

# evaluate the spectra and fill the RegionNDMap
m.quantity = newspec * u.cm**-2 * u.s**-1 * u.GeV**-1

m

Set the T0 of the event and save the gammapy model to a file:

In [ ]:
temporal_model_ref = Time(61946.23361111111444188, format="mjd", scale="utc")
filename = "./catO5_301_gammapy_format.fits"
temp = LightCurveTemplateTemporalModel(m, t_ref=temporal_model_ref, filename=filename, method="linear", values_scale="log")
#temp.method = "log"
#temp.values_scale = "log"
temp.write(filename, format="map", overwrite=True)

Re-open it and set how the times and spectra are interpolated (linear for time, log for spectra):

In [ ]:
temporal_model = LightCurveTemplateTemporalModel.read(filename, format="map")
temporal_model.method = "linear"  # default
temporal_model.values_scale = "log"  # default

Let's plot it to verify it's correct:

In [ ]:
#time_range = temporal_model.reference_time + [-10, 300] * u.s
#time_range = temporal_model.reference_time + [0.3, 2] * u.s
time_range = temporal_model.reference_time + [0.3, 300] * u.s

temporal_model.plot(time_range=time_range, energy=[0.0123, 0.05, 0.1, 0.2, 0.5, 1] * u.TeV, n_points=1000)
plt.semilogy()
plt.legend()
#plt.loglog()

# Set the inputs for the simulation

If you have the pointing plan and you want to take the same information:

In [ ]:
#obsid = "5000000204"
#obsid = "5000000230"
#obsid = "5000003012"
#obsid = "5000003185"
#obsid = "5000003220"
obsid = "5000003425"
pointing_plan = Table.read("./pointing_plan.fits", format="fits")
line = pointing_plan[pointing_plan["ObsID"] == obsid]
line

Let's define the main properties of the simulation (energy range, exposure, irf, pointing ecc):

In [ ]:
# main dir
path = Path("./")
output_folder = "event_sampling"
Path(path / f"{output_folder}").mkdir(exist_ok=True)

# pointing properties
pointing_coordinates = SkyCoord(ra=274.219, dec=1.492, unit="deg", frame="icrs")

# exposure time
livetime = 30.00000000000469 * u.s
#livetime = 60 * u.s

# irf and observatory location
irf_filename = ("Prod5-South-20deg-AverageAz-14MSTs37SSTs.18000s-v0.1.fits.gz")
location = observatory_locations["cta_south"]

# energy range (true and reco) for the simulation
e_min, e_max = 0.012589254, 199.52623
energy_axis = MapAxis.from_energy_bounds(f"{e_min} TeV", f"{e_max} TeV", nbin=10, per_decade=True)
energy_axis_true = MapAxis.from_energy_bounds(
    "0.001 TeV", "250 TeV", nbin=10, per_decade=True, name="energy_true"
)

# geomtry properties of the simulation
width = 2 # in deg
binsz = 0.01 # in deg

# reference time of the 1SDC
t_ref = Time("2028-01-01T00:00:00", format="isot", scale="utc")

# Make the sky-model for gammapy

In [ ]:
# define the sky model
spatial_model = PointSpatialModel.from_position(pointing_coordinates)
spectral_model = ConstantSpectralModel(const="1 cm-2 s-1 GeV-1")

model = SkyModel(
    spatial_model=spatial_model,
    spectral_model=spectral_model,
    temporal_model=temporal_model,
    name="GW",
)

#cosmic bkg from the IRF
bkg_model = FoVBackgroundModel(dataset_name="my-dataset")

#make the full model
models = Models([model, bkg_model])

if you need to evaluate the flux at a given time and energy:

In [ ]:
print(models)

# Prepare the dataset and simulate the events

In [ ]:
# telescope is pointing at a fixed position in ICRS for the observation
irf_path = path = Path("/Users/fabiopintore/LAVORO/CTA/DATA_CHALLENGE/sdc-simulations/MAKE_SDC/input/" +
                       "caldb/CTA-Performance-prod5-v0.1-South-20deg")

pointing = FixedPointingInfo(
    fixed_icrs=pointing_coordinates.icrs,
)

irfs = load_irf_dict_from_file(irf_path / irf_filename)

if you need to inspect the irf properties choose "aeff", "psf", "edisp", "bkg":

In [ ]:
# print(irfs["aeff"].axes["energy_true"])
# irfs["aeff"].peek()

In [ ]:
# # plot of the aeff on-axis
# plt.plot(irfs["aeff"].axes["energy_true"].center, irfs["aeff"].data[:,0])
# plt.loglog()
# plt.xlim(1e-2,3e2)
# plt.ylabel("Effective area [m2]")
# plt.xlabel("Energy [TeV]")

Let's define the gammapy observation object for the simulation:

In [ ]:
#tstart = Time("2028-06-24T05:40:24.000", format="isot", scale="utc")
#tstart = Time("2028-06-24T05:36:24.000", format="isot", scale="utc") + 0.3*u.s
#tstart = Time("2028-06-24T05:36:24.000", format="isot", scale="utc") + 60*u.s
#tstart = Time("2028-06-24T05:36:24.000", format="isot", scale="utc") + 120*u.s
tstart = Time("2028-06-24T05:40:24.000", format="isot", scale="utc")

observation = Observation.create(
    obs_id="0001",
    pointing=pointing,
    livetime=livetime,
    irfs=irfs,
    location=location,
    reference_time=t_ref,
    tstart=(tstart.utc.mjd-t_ref.utc.mjd) * u.d,
)
print(observation)

In [ ]:
observation.gti

Define a simulation geometry:

In [ ]:
migra_axis = MapAxis.from_bounds(0.5, 2, nbin=150, node_type="edges", name="migra")

geom = WcsGeom.create(
    skydir=pointing.fixed_icrs,
    width=(width, width),
    binsz=binsz,
    frame="icrs",
    axes=[energy_axis],
)

Create the MapDataset for the simulation:

In [ ]:
empty = MapDataset.create(
    geom,
    energy_axis_true=energy_axis_true,
    migra_axis=migra_axis,
    name="my-dataset",
)
maker = MapDatasetMaker(selection=["exposure", "background", "psf", "edisp"])
dataset = maker.run(empty, observation)

In [ ]:
dataset.models = models
print(dataset.models)

# Simulate the events

In [ ]:
sampler = MapDatasetEventSampler(random_state=0, oversample_energy_factor=10)
events = sampler.run(dataset, observation)

print(f"Source events: {(events.table['MC_ID'] == 1).sum()}")
print(f"Background events: {(events.table['MC_ID'] == 0).sum()}")

In [ ]:
events.table

In [ ]:
events.peek()